# Linear Algebra Practice

In [2]:
import numpy as np
import matplotlib.pyplot as plt

### Column space, null space, rank

In [3]:
def generate_rank_matrix(m, n, r, seed=None):
    rng = np.random.default_rng(seed)
    while True:
        U = rng.normal(size=(m, r))
        V = rng.normal(size=(r, n))
        A = U @ V
        if np.linalg.matrix_rank(A) == r:
            return A
A = generate_rank_matrix(3,3,2)

In [4]:
def rref(A, tol=1e-12, ignore_last=False):
    pivots = []
    row = 0
    for c in range(len(A[0])):
        #check which column has the pivot
        if c == len(A[0])-1 and ignore_last:
            continue
        pivot = None
        for r in range(row, len(A)):
            if abs(A[r, c]) > tol:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            A[[r, pivot]] = A[[pivot, r]]
        A[r] = A[r] / A[r][c]
        for r in range(len(A)):
            if r != row:
                A[r] -= A[row] * A[r][c]
        pivots.append(row)
        row += 1
        if row == len(A[0]):
            break
    A[abs(A) < tol] = 0.0
    return A, pivots


R, pivots = rref(A.copy())                

In [13]:
def build_nullspace(R, pivots, tol=1e-12):
    R = R.astype(float)
    m, n = R.shape
    pivots = list(pivots)
    free_cols = [j for j in range(n) if j not in pivots]
    r = len(pivots)
    f = len(free_cols)

    if f == 0:
        return np.zeros((n, 0))  # empty basis

    N = np.zeros((n, f))
    for k, free_j in enumerate(free_cols):
        x = np.zeros(n)
        x[free_j] = 1.0
        for i, pivot_j in enumerate(pivots):
            x[pivot_j] = -R[i, free_j]
        N[:, k] = x

    N[np.abs(N) < tol] = 0.0
    return N


N = build_nullspace(A.copy(), pivots)
print(N)

[[ 0.8451154 ]
 [-0.27775445]
 [ 1.        ]]


In [14]:
def column_basis(A, pivots):
    return A[:, pivots]
column_basis(A.copy(), pivots)

array([[-2.64189853,  0.91971441],
       [ 1.88643056, -2.41758817],
       [ 3.58266128, -2.89779971]])

In [15]:
def compute_solution(A, b, tol=1e-12):
    m, n = A.shape
    Aug = np.column_stack([A.astype(float), b.astype(float)])
    R_aug, pivots = rref(Aug, tol, True)  
    R = R_aug[:, :n]
    rhs = R_aug[:, n]

    for i in range(m):
        if np.all(np.abs(R[i]) < tol) and abs(rhs[i]) > tol:
            return np.zeros((n, 0)), "none"

    N = build_nullspace(R, pivots, tol)  
    x_p = np.zeros(n)
    for i, pcol in enumerate(pivots):
        if pcol < n:         
            x_p[pcol] = rhs[i]

    if N.shape[1] == 0:
        return x_p, "unique"
    else:
        full = np.column_stack([x_p, N])  
        return full, "infinite"
b = np.sum(column_basis(A.copy(), pivots), axis=1)
compute_solution(A, b)

(array([[ 1.        , -0.38427963],
        [ 1.        , -0.18496218],
        [ 0.        ,  1.        ]]),
 'infinite')

In [16]:
def compute_projections(a, b):
    numerator = np.dot(a.T, b)
    denominator = np.dot(a.T, a)
    x = numerator/denominator
    p = a * x
    return p
a = np.random.uniform(0, 10, size=3)
b = np.random.uniform(0, 10, size=3)
p = compute_projections(a,b)
ortho = np.dot((b-p), a)
print(p, ortho.round(10))

[3.3971233  7.44011357 0.63068935] 0.0


In [19]:
def compute_projection_matrix(A):
    _, pivots = rref(A)
    A = A[:, pivots]
    inverse = np.linalg.inv(A.T @ A)
    P = A @ inverse @ A.T
    P[np.abs(P) < 1e-12] = 0.0
    return P


P = compute_projection_matrix(A)
projection = P @ b
print("Projection matrix P:")
print(P)
print("\nProjection of b onto column space of A:")
print(projection)
# Verify idempotent property of projection matrix
print("P @ P == P:", np.allclose(P @ P, P))

# Check orthogonality: A^T * (projection - b) should be zero
orthogonal_check = A.T @ (projection - b)
print("\nA^T * (projection - b):")
print(orthogonal_check)
print("Is orthogonal (close to zero):", np.allclose(orthogonal_check, 0))

Projection matrix P:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 0.]]

Projection of b onto column space of A:
[1.12722148 8.17169332 0.        ]
P @ P == P: True

A^T * (projection - b):
[0. 0. 0.]
Is orthogonal (close to zero): True
